# Layer‑Wise Residual Stream Ablation
Tests whether the typosquat signal is carried by the residual stream by zeroing out entire layer contributions.

In [ ]:
!pip install -q torch transformers scikit-learn matplotlib seaborn

In [ ]:
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModelForCausalLM
import json, re, os
from tqdm import tqdm

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
DATASET_PATH = "typosquat_dataset_full/typosquat_tool_calls.jsonl"
df_all = pd.DataFrame([json.loads(line) for line in open(DATASET_PATH)])
df_adv = df_all[df_all['is_adversarial'] == True].sample(n=300, random_state=SEED).copy()
print(f"Sampled {len(df_adv)} adversarial entries")

texts, labels = [], []
for _, row in df_adv.iterrows():
    texts.append(row['clean_command']); labels.append(0)
    texts.append(row['typo_command']); labels.append(1)

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model.eval()
num_layers = model.config.num_hidden_layers
print(f"Model has {num_layers} transformer layers")

In [ ]:
def extract_features(model, tokenizer, texts, batch_size=16):
    feats = []
    for i in range(0, len(texts), batch_size):
        batch = tokenizer(texts[i:i+batch_size], padding=True, truncation=True, max_length=64, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outs = model(**batch, output_hidden_states=True)
            feats.append(outs.hidden_states[-1].mean(dim=1).float().cpu().numpy())
    return np.vstack(feats)

X_base = extract_features(model, tokenizer, texts)
y = np.array(labels)
probe_base = LogisticRegression(max_iter=1000, random_state=SEED).fit(X_base, y)
auc_base = roc_auc_score(y, probe_base.predict_proba(X_base)[:, 1])
print(f"Baseline AUC: {auc_base:.4f}")

In [ ]:
def extract_with_layer_ablation(ablated_layers):
    hooks = []
    def make_zero_hook():
        def hook_fn(module, input, output):
            return torch.zeros_like(output)
        return hook_fn
    for l_idx in ablated_layers:
        layer = model.model.layers[l_idx]
        hook = layer.register_forward_hook(make_zero_hook())
        hooks.append(hook)
    feats = []
    for i in range(0, len(texts), 8):
        batch = tokenizer(texts[i:i+8], padding=True, truncation=True, max_length=64, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outs = model(**batch, output_hidden_states=True)
            feats.append(outs.hidden_states[-1].mean(dim=1).float().cpu().numpy())
    for hook in hooks:
        hook.remove()
    return np.vstack(feats)

print("\n=== Single-Layer Ablation Sweep ===")
results = []
for l in range(num_layers):
    X_abl = extract_with_layer_ablation([l])
    probe_abl = LogisticRegression(max_iter=1000, random_state=SEED).fit(X_abl, y)
    auc_abl = roc_auc_score(y, probe_abl.predict_proba(X_abl)[:, 1])
    drop = auc_base - auc_abl
    results.append({'layer': l, 'auc': auc_abl, 'drop': drop})
    print(f"L{l:2d}: AUC={auc_abl:.4f} (Δ={drop:+.4f})")

res_df = pd.DataFrame(results)

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(res_df['layer'], res_df['auc'], 'o-', linewidth=2, markersize=6)
plt.axhline(y=auc_base, ls='--', c='gray', label=f'Baseline AUC ({auc_base:.3f})')
plt.xlabel('Ablated Layer Index')
plt.ylabel('Probe AUC After Ablation')
plt.title('Causal Impact of Single-Layer Residual Ablation on Typosquat Detection')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('layer_ablation_curve.png', dpi=150)
plt.show()

In [ ]:
critical = res_df[res_df['drop'] > 0.02]
if len(critical) > 0:
    print(f"\n✅ Critical layers (ΔAUC > 0.02): {list(critical['layer'])}")
else:
    print("\n⚠️ No single layer causes meaningful AUC drop. Signal is redundantly propagated.")